# 🧠 ISOM 260: Train Your Own Tiny LLM

**Session 1 — Welcome to the AI Revolution** | Suffolk University | Prof. Hasan Arslan

---

Six days ago, OpenAI shipped GPT-6. Today, in the next 40 minutes, **you** are going to train a language model — a real one, from scratch, in your browser, for free.

It will be tiny. But here's the secret this whole course is built on:

> **Every language model — from the toy we build today to GPT-6 — does exactly one thing: predict the next token.**

Everything else is scale.

### What you'll do
1. **Act 1 — The Counting Model.** Build a language model with nothing but counting. No math beyond tallying.
2. **Act 2 — The Transformer.** Train a real miniature GPT on the same data and watch the quality jump.
3. **Act 3 — The Scale Story.** See exactly what separates your model from GPT-6. (Spoiler: less than you think.)

### How to run this notebook
- Click a cell, press **Shift + Enter** to run it. Run cells **top to bottom**.
- No API keys. No installs. No experience needed.
- **First:** go to `File → Save a copy in Drive` so you have your own copy.

*(This exercise is inspired by Andrej Karpathy's legendary [makemore](https://github.com/karpathy/makemore) and [nanochat](https://github.com/karpathy/nanochat) — the "build it yourself and it stops being magic" school of AI education.)*


## 🎮 Warm-up: You are a language model

Complete this sentence: **"The capital of France is ____"**

You said *Paris*. How? You've seen that pattern so many times that the next word is obvious. You didn't "look it up" — you *predicted* it.

That's the entire job of a language model: **given some text, predict what comes next.** Do that well enough, over enough text, and you get something that looks like intelligence.

Our mission today: build a model that learns to **invent new human names**. It will read 32,000 real names, learn the patterns, and generate names that *don't exist* but *sound real* — the same way ChatGPT generates sentences that were never written but sound right.


In [ ]:
# ── Load the data: 32,033 real names ──────────────────────────────
# (dataset from Andrej Karpathy's makemore project)
import urllib.request
import random
random.seed(260)  # ISOM 260 — makes everyone's results reproducible

url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
names = urllib.request.urlopen(url).read().decode().splitlines()

print(f"Loaded {len(names):,} names.")
print("First 10:", names[:10])
print("A random sample:", random.sample(names, 5))

## 🎬 Act 1 — The Counting Model

Here's the world's simplest language model, and you already know how to build it:

> **For every letter, count which letter tends to come next.**

That's it. If after `q` we almost always see `u`, then when generating, after a `q` we should probably output a `u`. We'll use a special symbol `.` to mean *"start of name"* and *"end of name"* — so the model also learns which letters names tend to **start** and **end** with.

A model that only looks at **pairs** of letters is called a **bigram model**. Let's count.


In [ ]:
# ── Count every letter pair in 32,033 names ───────────────────────
counts = {}  # counts[a][b] = how many times letter b followed letter a

for name in names:
    letters = ["."] + list(name) + ["."]   # "emma" -> . e m m a .
    for a, b in zip(letters, letters[1:]):
        counts.setdefault(a, {}).setdefault(b, 0)
        counts[a][b] += 1

# What did it learn? Let's interrogate the model.
def top_next(letter, k=5):
    ranked = sorted(counts[letter].items(), key=lambda kv: -kv[1])[:k]
    total = sum(counts[letter].values())
    return ", ".join(f"'{b}' ({100*c/total:.0f}%)" for b, c in ranked)

print("After 'q', the next letter is usually:", top_next("q"))
print("Names most often START with:         ", top_next("."))
print("After 'a', the next letter is usually:", top_next("a"))
print("Names most often END after... try it — pick your own letter below!")
print("After 'x':", top_next("x"))

**Look at that.** The model discovered — purely by counting — that `q` is almost always followed by `u`, and that names love to start with `a`, `k`, and `m`. Nobody programmed those rules. They emerged from data.

This is the core move of all machine learning: **patterns in, behavior out.**

Now the fun part — let's make it *generate*. To invent a name: start at `.`, roll a weighted die to pick the next letter, then roll again from *that* letter... until we hit `.` again.


In [ ]:
# ── Generate brand-new names by rolling weighted dice ─────────────
def generate_bigram(temperature=1.0):
    out, letter = [], "."
    while True:
        nxt = counts[letter]
        options = list(nxt.keys())
        # temperature: <1 = play it safe, >1 = get weird (more on this below)
        weights = [c ** (1.0 / temperature) for c in nxt.values()]
        letter = random.choices(options, weights=weights)[0]
        if letter == ".":
            return "".join(out)
        out.append(letter)

print("✨ 15 names invented by the counting model:\n")
for _ in range(15):
    print("  ", generate_bigram().capitalize())

### 🤔 Discussion (30 seconds with your neighbor)

Some of those *almost* work... and some are keyboard-smash. Why?

Because our model has **one letter of memory**. When it's picking letter #6, it only remembers letter #5. It can't know it's already used three vowels in a row, or that the name is getting long. Every step, it forgets everything except the last letter.

Hold that thought — it's exactly what the transformer will fix.

### 🌡️ First, play with the creativity dial

That `temperature` knob is **the same temperature setting** you'll see in the ChatGPT/Claude/Gemini APIs in Session 3. Low = safe and repetitive. High = creative and unhinged.


In [ ]:
# ── Temperature: the creativity dial ──────────────────────────────
for t in [0.5, 1.0, 2.0]:
    sample = [generate_bigram(temperature=t).capitalize() for _ in range(6)]
    label = {0.5: "😴 T=0.5 (cautious)", 1.0: "🙂 T=1.0 (normal)  ", 2.0: "🤪 T=2.0 (chaotic) "}[t]
    print(label, "→", ", ".join(sample))

## 🎬 Act 2 — The Transformer

In 2017, Google researchers published a paper called *"Attention Is All You Need."* It introduced the **transformer** — the architecture behind every modern AI you've heard of. The **T in GPT stands for Transformer.**

The breakthrough, in one sentence:

> Instead of remembering only the previous letter, **attention** lets the model look back at *everything* written so far — and *learn which parts matter* for predicting what comes next.

Below is a complete, real transformer — the same architecture family as GPT-6, shrunk to ~100,000 parameters. **You do not need to read this code.** Skim it, run it, and appreciate that the entire "secret" of modern AI fits in one scrollable cell.


In [ ]:
# ── A complete miniature GPT (~100K parameters) ───────────────────
# You don't need to understand this cell — run it and skim.
# It's the same architecture family as GPT-6, at 1/10,000,000th the size.
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(260)

# Data: one long stream of names separated by newlines
text = "\n".join(names)
chars = sorted(set(text))                     # our 27-symbol "vocabulary"
stoi = {c: i for i, c in enumerate(chars)}    # letter -> number
itos = {i: c for c, i in stoi.items()}        # number -> letter
data = torch.tensor([stoi[c] for c in text])

# Hyperparameters (the "size dials" — GPT-6 turns these WAY up)
block_size, n_embd, n_head, n_layer = 16, 64, 4, 2
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(batch_size=64):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(),
                                 nn.Linear(4*n_embd, n_embd))
    def forward(self, x):
        T = x.shape[1]  # causal mask: you can't peek at the future
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(len(chars), n_embd)   # what each letter "means"
        self.pos = nn.Embedding(block_size, n_embd)   # where it sits in the sequence
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.head = nn.Linear(n_embd, len(chars))     # scores for the next letter
    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.blocks(x))

model = TinyGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built on {device.upper()} — {n_params:,} parameters.")
print(f"(GPT-4-class models are ~10,000,000× bigger. Same idea.)")

In [ ]:
# ── Train it: watch a language model learn, live ──────────────────
# ~1–3 minutes. The 'loss' is how SURPRISED the model is by real names.
# Watch it fall: falling loss = the model is literally learning English
# name patterns before your eyes.
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(2501):
    x, y = get_batch()
    loss = F.cross_entropy(model(x).view(-1, len(chars)), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 250 == 0:
        print(f"step {step:5d} | surprise (loss): {loss.item():.3f}")
print("\n🎓 Training complete. Your transformer has learned... names.")

In [ ]:
# ── The showdown: counting model vs. YOUR transformer ─────────────
@torch.no_grad()
def generate_gpt(prefix="", n=10, temperature=1.0):
    model.eval()
    results = []
    while len(results) < n:
        ctx = torch.tensor([[stoi["\n"]] + [stoi[c] for c in prefix]], device=device)
        for _ in range(20):
            logits = model(ctx[:, -block_size:])[0, -1] / temperature
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
            if itos[nxt.item()] == "\n":
                break
            ctx = torch.cat([ctx, nxt.view(1, 1)], dim=1)
        name = "".join(itos[i] for i in ctx[0, 1:].tolist())
        if len(name) > len(prefix) + 1:
            results.append(name.capitalize())
    return results

bigram_names = [generate_bigram().capitalize() for _ in range(10)]
gpt_names = generate_gpt(n=10)

print(f"{'ACT 1: Counting model':<28}ACT 2: Your transformer")
print("─" * 52)
for b, g in zip(bigram_names, gpt_names):
    print(f"{b:<28}{g}")

**Same data. Same goal. Radically better output.** The only difference: the transformer can *pay attention* to the whole name so far, not just the last letter.

### 🎨 Your turn: brand the model

Give the model the start of a name and let it finish. Try your initials, your own name's first 3 letters, or invent a product line for a startup:


In [ ]:
# ── Seed it with your own prefix — edit and re-run! ───────────────
your_prefix = "suf"   # ← change me (lowercase letters only)

print(f"Names starting with '{your_prefix}':\n")
for name in generate_gpt(prefix=your_prefix, n=8):
    print("  ", name)

## 🎬 Act 3 — The Scale Story

You just trained a transformer. So what's the difference between your model and the ones that shipped last week?

| | **Your TinyGPT** | **GPT-2 (2019)** | **GPT-4 class (2023)** | **GPT-6 Astra (last week)** |
|---|---|---|---|---|
| Parameters | ~100 thousand | 1.5 billion | ~1.8 trillion | undisclosed (frontier) |
| Training data | 32,033 names | 8M web pages | most of the internet | + synthetic & multimodal |
| Training cost | $0 (free Colab) | ~$50K | ~$100M+ | ~$1B+ (estimated) |
| Training time | ~2 minutes | weeks | months | months |
| Objective | **predict the next token** | **predict the next token** | **predict the next token** | **predict the next token** |

Read that last row again. **The objective never changed.** What changed is scale — plus three additions you'll meet later this semester:

1. **Instruction tuning & RLHF** *(Session 2)* — teaching the raw predictor to be a helpful assistant instead of an autocomplete.
2. **Tools & agents** *(Sessions 4–8)* — letting the model search, calculate, call APIs, and act. This is where the business value lives, and it's most of this course.
3. **Retrieval / RAG** *(Session 7)* — connecting the model to *your* documents and data.

### 🧭 Reflection (bring answers to Session 2)
1. Your model invents names. ChatGPT "invents" sentences the same way. What does that tell you about why models sometimes **hallucinate** — confidently make things up?
2. Temperature made output safer or wilder. For which **business tasks** would you want T low? T high?
3. Your model learned *only* from the names it saw. What are the business risks of a model that can only reflect its **training data**?

---

### ✅ Before you leave today
- `File → Save a copy in Drive` (if you haven't)
- Post your **single favorite generated name** in the Canvas Session 1 thread 🏆
- Bookmark [aistudio.google.com](https://aistudio.google.com) — you'll get an API key there in Session 3

**Next session:** *How AI Actually Works* — we open the hood on the thing you just built: neurons, attention, and how a next-letter predictor became a trillion-dollar industry.

*You trained a language model on day one. Most people never will. Welcome to ISOM 260.* 🚀
